# LeetCode 3538 — Merge Operations for Minimum Travel Time bằng Python

- Link: https://leetcode.com/problems/merge-operations-for-minimum-travel-time/
- Độ khó: **Hard**
- Chủ đề: **Quy hoạch động** trên dãy với trạng thái `(biển giữ lại cuối, biển giữ lại trước đó, số lần gộp)`, **tổng tiền tố**, mô hình hóa "khối bị xóa"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minTravelTime(self, l: int, n: int, k: int, position: List[int], time: List[int]) -> int:
```

## 1. Mục tiêu học

1. Hiểu phép gộp: xóa biển `i` và **cộng dồn** `time[i]` vào biển `i+1` ⇒ sau nhiều lần gộp, mỗi biển còn lại mang **tổng thời gian của khối biển bị xóa ngay trước nó** cộng thời gian của chính nó.
2. Nhận ra kết quả chỉ phụ thuộc **tập biển bị xóa**, không phụ thuộc thứ tự gộp.
3. Thiết kế DP `dp[j][i][c]`: biển giữ lại cuối là `j`, biển giữ lại trước nó là `i` (quyết định `time` hiện tại của `j`), đã xóa `c` biển.
4. Dùng **tổng tiền tố** để lấy `time` gộp `= pre[j+1] − pre[i+1]` trong `O(1)`.
5. Kiểm chứng bằng mô phỏng **mọi dãy thao tác gộp** thật sự trên dữ liệu nhỏ.

## 2. Đề bài & ví dụ

Con đường dài `l` km, có `n` biển ở các vị trí `position` tăng ngặt (`position[0] = 0`, `position[n−1] = l`). `time[i]` là số phút đi **1 km** trên đoạn từ biển `i` tới biển `i+1`.

Phải thực hiện **đúng `k`** lần gộp. Mỗi lần chọn hai biển kề nhau `i`, `i+1` với `i > 0` và `i + 1 < n`:

- `time[i+1] = time[i] + time[i+1]`;
- xóa biển `i`.

Trả về tổng thời gian đi từ `0` tới `l` nhỏ nhất sau đúng `k` lần gộp.

**Ví dụ 1:** `l=10, n=4, k=1, position=[0,3,8,10], time=[5,8,3,6]` → `62` (gộp 1 và 2: vị trí `[0,8,10]`, time `[5,11,6]`: `8·5 + 2·11 = 62`).

**Ví dụ 2:** `l=5, n=5, k=1, position=[0,1,2,3,5], time=[8,3,9,3,3]` → `34`.

**Ràng buộc:** `2 ≤ n ≤ min(l+1, 50)`, `0 ≤ k ≤ min(n−2, 10)`, `1 ≤ time[i] ≤ 100`, `Σ time ≤ 100`.

In [1]:
from typing import List
examples = [
    (10, 4, 1, [0, 3, 8, 10], [5, 8, 3, 6], 62),
    (5, 5, 1, [0, 1, 2, 3, 5], [8, 3, 9, 3, 3], 34),
]
for l, n, k, pos, tm, exp in examples:
    print(f"l={l}, n={n}, k={k}, position={pos}, time={tm} -> kỳ vọng {exp}")

l=10, n=4, k=1, position=[0, 3, 8, 10], time=[5, 8, 3, 6] -> kỳ vọng 62
l=5, n=5, k=1, position=[0, 1, 2, 3, 5], time=[8, 3, 9, 3, 3] -> kỳ vọng 34


## 3. Kiến thức nền

### 3.1. Tổng thời gian đi đường

Với các biển còn lại `p_0 < p_1 < … < p_r` và thời gian `t_0, …, t_r`: tổng `= Σ (p_{s+1} − p_s) · t_s` (biển cuối không dùng thời gian của nó).

In [2]:
def travel(pos, tm):
    return sum((pos[s + 1] - pos[s]) * tm[s] for s in range(len(pos) - 1))

print(travel([0, 8, 10], [5, 11, 6]), travel([0, 3, 8, 10], [5, 8, 3, 6]))

62 61


### 3.2. Mô phỏng một lần gộp

In [3]:
def merge_at(pos, tm, i):
    assert 0 < i and i + 1 < len(pos)
    tm2 = tm[:i] + [tm[i] + tm[i + 1]] + tm[i + 2:]
    pos2 = pos[:i] + pos[i + 1:]
    return pos2, tm2

print(merge_at([0, 3, 8, 10], [5, 8, 3, 6], 1))

([0, 8, 10], [5, 11, 6])


### 3.3. Tổng tiền tố

`pre[j] = time[0] + … + time[j−1]` ⇒ tổng `time[a..b] = pre[b+1] − pre[a]`.

In [4]:
tm = [8, 3, 9, 3, 3]
pre = [0]
for t in tm:
    pre.append(pre[-1] + t)
print('pre =', pre, '| time[1..2] =', pre[3] - pre[1])

pre = [0, 8, 11, 20, 23, 26] | time[1..2] = 12


### 3.4. Thứ tự gộp không quan trọng

Mỗi biển bị xóa "đẩy" thời gian hiện tại của nó sang biển kề phải. Nếu một khối liên tiếp `i+1, …, j−1` bị xóa (và `i`, `j` được giữ) thì cuối cùng biển `j` mang `time[i+1] + … + time[j]` — bất kể xóa theo thứ tự nào. Ta kiểm chứng bằng code ở mục 5.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Thay vì nghĩ "thao tác gộp", hãy nghĩ "**chọn `k` biển bên trong để xóa**". Với tập biển giữ lại `0 = j_0 < j_1 < … < j_r = n−1`:

- thời gian mới của `j_s` là `T(j_s) = time[j_{s−1}+1] + … + time[j_s]` (với `j_0 = 0`: `T = time[0]`);
- tổng = `Σ (position[j_{s+1}] − position[j_s]) · T(j_s)`.

Tính tay ví dụ 1 (`k = 1`, xóa một trong hai biển 1, 2):

| xóa | giữ | time mới | tổng |
|---|---|---|---|
| 1 | 0, 2, 3 | 5, 8+3 = 11, 6 | 8·5 + 2·11 = 62 |
| 2 | 0, 1, 3 | 5, 8, 3+6 = 9 | 3·5 + 7·8 = 71 |

⇒ `62`.

In [5]:
def cost_of_kept(position, time, kept):
    total, prev = 0, None
    for s in range(len(kept) - 1):
        j = kept[s]
        T = time[0] if s == 0 else sum(time[kept[s - 1] + 1: j + 1])
        total += (position[kept[s + 1]] - position[j]) * T
    return total

pos, tm = [0, 3, 8, 10], [5, 8, 3, 6]
for kept in ([0, 2, 3], [0, 1, 3]):
    print('giữ', kept, '->', cost_of_kept(pos, tm, kept))

giữ [0, 2, 3] -> 62
giữ [0, 1, 3] -> 71


## 5. Cách 1 — Brute Force: mô phỏng mọi dãy `k` thao tác gộp

Đệ quy: ở mỗi bước thử mọi vị trí gộp hợp lệ `i ∈ [1, n−2]`, sau `k` bước tính tổng thời gian. Đúng nguyên văn đề bài (không dùng suy luận nào) ⇒ oracle.

In [6]:
def min_travel_simulate(l, n, k, position, time):
    best = [float('inf')]

    def rec(pos, tm, left):
        if left == 0:
            best[0] = min(best[0], travel(pos, tm))
            return
        for i in range(1, len(pos) - 1):
            p2, t2 = merge_at(pos, tm, i)
            rec(p2, t2, left - 1)

    rec(position, time, k)
    return best[0]

for l, n, k, pos, tm, exp in examples:
    print(min_travel_simulate(l, n, k, pos, tm), '| kỳ vọng', exp)

62 | kỳ vọng 62
34 | kỳ vọng 34


**Độ phức tạp:** `(n−2)(n−3)…(n−1−k)` dãy thao tác — với `n = 50, k = 10` là ~`10^{16}`.

**Nút thắt:** nhiều dãy thao tác cho cùng tập biển bị xóa (thứ tự không quan trọng) ⇒ chỉ cần chọn **tập**.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề:** kết quả chỉ phụ thuộc tập biển bị xóa `R ⊆ {1, …, n−2}`, `|R| = k`, và được tính bằng công thức ở mục 4.

**DP:** đi qua các biển giữ lại từ trái sang phải. Trạng thái `(j, i, c)`:

- `j`: biển giữ lại **hiện tại** (cuối cùng đã chọn);
- `i`: biển giữ lại **trước** `j` (để biết `T(j) = pre[j+1] − pre[i+1]`; với `j = 0` quy ước `i = −1` ⇒ `T = time[0]`);
- `c`: số biển đã xóa (= các biển nằm giữa các biển giữ lại liên tiếp).

`dp[j][i][c]` = tổng thời gian nhỏ nhất của quãng `0 → position[j]`.

Chuyển sang biển giữ lại tiếp theo `t > j` (xóa `t − j − 1` biển ở giữa):

`dp[t][j][c + t − j − 1] = min(…, dp[j][i][c] + (position[t] − position[j]) · T(j))`.

Đáp án: `min_i dp[n−1][i][k]`.

Quy tắc ngắn:

- **Nếu** giữ `t` ngay sau `j` **thì** khối `j+1..t−1` bị xóa và `T(t) = Σ time[j+1..t]`.
- **Nếu** `c` vượt `k` **thì** bỏ.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Thử mọi tập `k` biển bị xóa

In [7]:
from itertools import combinations

def min_travel_subsets(l, n, k, position, time):
    best = float('inf')
    for removed in combinations(range(1, n - 1), k):
        rs = set(removed)
        kept = [j for j in range(n) if j not in rs]
        best = min(best, cost_of_kept(position, time, kept))
    return best

for l, n, k, pos, tm, exp in examples:
    print(min_travel_subsets(l, n, k, pos, tm), '| kỳ vọng', exp)

62 | kỳ vọng 62
34 | kỳ vọng 34


**Độ phức tạp:** `C(n−2, k)` tập — với `n = 50, k = 10` khoảng `5·10^9`: vẫn quá lớn.

### 7.2. Cách 3 — DP có nhớ (top-down)

`go(j, i, c)` = chi phí nhỏ nhất **từ** biển `j` tới cuối đường, biết biển giữ lại trước `j` là `i` và đã xóa `c` biển.

In [8]:
from functools import lru_cache

def min_travel_memo(l, n, k, position, time):
    pre = [0]
    for t in time:
        pre.append(pre[-1] + t)

    @lru_cache(maxsize=None)
    def go(j, i, c):
        if j == n - 1:
            return 0 if c == k else float('inf')
        T = pre[j + 1] - pre[i + 1]           # i = -1 khi j = 0 -> T = time[0]
        best = float('inf')
        for t in range(j + 1, n):
            nc = c + (t - j - 1)
            if nc > k:
                break
            best = min(best, (position[t] - position[j]) * T + go(t, j, nc))
        return best

    return go(0, -1, 0)

for l, n, k, pos, tm, exp in examples:
    print(min_travel_memo(l, n, k, pos, tm), '| kỳ vọng', exp)

62 | kỳ vọng 62
34 | kỳ vọng 34


Chú ý `pre[i + 1]` với `i = −1` là `pre[0] = 0` ⇒ `T = pre[1] = time[0]` — đúng.

### 7.3. Cách 4 — DP bottom-up (lời giải chính)

Duyệt `j` tăng dần; `dp[j]` là dict `(i, c) → chi phí`. Mỗi bước `t − j − 1 ≤ k` nên chỉ cần thử `t ≤ j + k + 1`.

In [9]:
def min_travel_dp(l, n, k, position, time):
    pre = [0]
    for t in time:
        pre.append(pre[-1] + t)
    INF = float('inf')
    dp = [dict() for _ in range(n)]
    dp[0][(-1, 0)] = 0
    for j in range(n - 1):
        for (i, c), v in dp[j].items():
            T = pre[j + 1] - pre[i + 1]
            for t in range(j + 1, min(n, j + k - c + 2)):
                nc = c + t - j - 1
                cost = v + (position[t] - position[j]) * T
                key = (j, nc)
                if cost < dp[t].get(key, INF):
                    dp[t][key] = cost
    return min(v for (i, c), v in dp[n - 1].items() if c == k)

for l, n, k, pos, tm, exp in examples:
    print(min_travel_dp(l, n, k, pos, tm), '| kỳ vọng', exp)

62 | kỳ vọng 62
34 | kỳ vọng 34


**Độ phức tạp:** số trạng thái `≤ n · (k+1) · (k+1)` (vì `j − i − 1 ≤ k` ⇒ `i` chỉ có `k + 1` lựa chọn) và mỗi trạng thái thử `≤ k + 1` bước ⇒ `O(n · k^3)` ≈ `50 · 11^3 ≈ 7·10^4`. Rất nhanh.

## 8. TRACE từng bước

Trace cách 4 trên ví dụ 2: in các trạng thái `(i, c) → chi phí` tại mỗi biển `j` và chuyển trạng thái tốt nhất.

In [10]:
def trace(l, n, k, position, time):
    pre = [0]
    for t in time:
        pre.append(pre[-1] + t)
    INF = float('inf')
    dp = [dict() for _ in range(n)]
    dp[0][(-1, 0)] = 0
    for j in range(n - 1):
        print(f"biển {j} (vị trí {position[j]}): trạng thái (biển trước, số đã xóa) -> chi phí = {dp[j]}")
        for (i, c), v in dp[j].items():
            T = pre[j + 1] - pre[i + 1]
            for t in range(j + 1, min(n, j + k - c + 2)):
                nc = c + t - j - 1
                cost = v + (position[t] - position[j]) * T
                print(f"    từ (i={i}, c={c}) với T({j}) = {T}: giữ biển {t} (xóa {t - j - 1}) -> chi phí {cost}")
                if cost < dp[t].get((j, nc), INF):
                    dp[t][(j, nc)] = cost
    print(f"biển cuối {n - 1}: {dp[n - 1]}")
    ans = min(v for (i, c), v in dp[n - 1].items() if c == k)
    print('Đáp án (đúng k lần xóa) =', ans)
    return ans

trace(*examples[1][:5])

biển 0 (vị trí 0): trạng thái (biển trước, số đã xóa) -> chi phí = {(-1, 0): 0}
    từ (i=-1, c=0) với T(0) = 8: giữ biển 1 (xóa 0) -> chi phí 8
    từ (i=-1, c=0) với T(0) = 8: giữ biển 2 (xóa 1) -> chi phí 16
biển 1 (vị trí 1): trạng thái (biển trước, số đã xóa) -> chi phí = {(0, 0): 8}
    từ (i=0, c=0) với T(1) = 3: giữ biển 2 (xóa 0) -> chi phí 11
    từ (i=0, c=0) với T(1) = 3: giữ biển 3 (xóa 1) -> chi phí 14
biển 2 (vị trí 2): trạng thái (biển trước, số đã xóa) -> chi phí = {(0, 1): 16, (1, 0): 11}
    từ (i=0, c=1) với T(2) = 12: giữ biển 3 (xóa 0) -> chi phí 28
    từ (i=1, c=0) với T(2) = 9: giữ biển 3 (xóa 0) -> chi phí 20
    từ (i=1, c=0) với T(2) = 9: giữ biển 4 (xóa 1) -> chi phí 38
biển 3 (vị trí 3): trạng thái (biển trước, số đã xóa) -> chi phí = {(1, 1): 14, (2, 1): 28, (2, 0): 20}
    từ (i=1, c=1) với T(3) = 12: giữ biển 4 (xóa 0) -> chi phí 38
    từ (i=2, c=1) với T(3) = 3: giữ biển 4 (xóa 0) -> chi phí 34
    từ (i=2, c=0) với T(3) = 3: giữ biển 4 (xóa 0) -> chi

34

## 9. Code nộp LeetCode

In [11]:
class Solution:
    def minTravelTime(self, l: int, n: int, k: int, position: List[int], time: List[int]) -> int:
        pre = [0]
        for t in time:
            pre.append(pre[-1] + t)
        INF = float('inf')
        dp = [dict() for _ in range(n)]
        dp[0][(-1, 0)] = 0
        for j in range(n - 1):
            pj = position[j]
            for (i, c), v in dp[j].items():
                T = pre[j + 1] - pre[i + 1]
                for t in range(j + 1, min(n, j + k - c + 2)):
                    key = (j, c + t - j - 1)
                    cost = v + (position[t] - pj) * T
                    if cost < dp[t].get(key, INF):
                        dp[t][key] = cost
        return min(v for (i, c), v in dp[n - 1].items() if c == k)

for l, n, k, pos, tm, exp in examples:
    got = Solution().minTravelTime(l, n, k, pos, tm)
    print(got, 'OK' if got == exp else 'SAI')

62 OK
34 OK


## 10. Vì sao thuật toán đúng

1. **Bất biến của phép gộp:** tại mọi thời điểm, mỗi biển còn lại `j` mang thời gian bằng tổng `time` gốc của `j` và mọi biển đã bị xóa nằm giữa biển còn lại liền trước và `j`. Đúng ban đầu (khối rỗng); gộp `i` vào `i+1` nối khối của `i` (kể cả `i`) vào trước khối của `i+1` ⇒ bất biến giữ nguyên.
2. **Phụ thuộc vào tập:** từ bất biến, trạng thái cuối chỉ phụ thuộc tập biển bị xóa. Mọi tập `R ⊆ {1..n−2}` cỡ `k` đều đạt được (xóa lần lượt từ trái sang phải).
3. **DP đúng:** một tập biển giữ lại ↔ một đường đi `0 = j_0 → j_1 → … → n−1` trong DP; chi phí cạnh `j_s → j_{s+1}` bằng `(position[j_{s+1}] − position[j_s])·T(j_s)`, với `T(j_s)` xác định bởi `j_{s−1}` (lưu trong trạng thái). Số biển bị xóa = tổng các khoảng trống ⇒ lọc `c = k`.
4. **Biển cuối** không cần thời gian; biển đầu (`0`) không bị xóa được và mang `time[0]`.

## 11. Phân tích Time / Space Complexity

Gọi `n ≤ 50`, `k ≤ 10`.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng mọi dãy thao tác | `O((n−2)^k · n)` | `O(k n)` |
| 2. Thử mọi tập bị xóa | `O(C(n−2, k) · n)` | `O(n)` |
| 3. DP có nhớ | `O(n · n · k · n)` (không cắt) | `O(n^2 k)` |
| 4. DP bottom-up có cắt | `O(n · k^3)` | `O(n · k^2)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {
    'subsets': min_travel_subsets,
    'memo': min_travel_memo,
    'dp': min_travel_dp,
    'submit': lambda *a: Solution().minTravelTime(*a),
}
edge = [
    (1, 2, 0, [0, 1], [7, 3], 7),                         # không có biển bên trong
    (6, 3, 0, [0, 2, 6], [1, 5, 9], 2 * 1 + 4 * 5),        # k = 0
    (6, 3, 1, [0, 2, 6], [1, 5, 9], 6 * 1),                # xóa biển 1 (time dồn sang biển cuối, không dùng)
    (10, 5, 3, [0, 1, 2, 3, 10], [1, 1, 1, 1, 1], 10),     # xóa hết biển bên trong
]
for l, n, k, pos, tm, exp in examples + edge:
    for name, f in impls.items():
        got = f(l, n, k, pos, tm)
        assert got == exp, (name, l, n, k, pos, tm, got, exp)
    assert min_travel_simulate(l, n, k, pos, tm) == exp
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với mô phỏng nguyên văn

In [13]:
import random
random.seed(3538)

def rand_case(n, k):
    l = random.randint(n - 1, 3 * n)
    inner = sorted(random.sample(range(1, l), n - 2))
    pos = [0] + inner + [l]
    tm = [random.randint(1, 9) for _ in range(n)]
    return l, n, k, pos, tm

cnt = 0
for _ in range(400):
    n = random.randint(2, 7)
    k = random.randint(0, min(n - 2, 3))
    case = rand_case(n, k)
    exp = min_travel_simulate(*case)
    for name, f in impls.items():
        assert f(*case) == exp, (name, case)
    cnt += 1
print('Khớp mô phỏng trên', cnt, 'bộ test ngẫu nhiên.')

Khớp mô phỏng trên 400 bộ test ngẫu nhiên.


### 12.3. Ca lớn (`n = 50`, `k = 10`, `Σ time ≤ 100`)

In [14]:
import time as _time
random.seed(50)
for trial in range(3):
    n, k = 50, 10
    l = random.randint(49, 10**5)
    pos = [0] + sorted(random.sample(range(1, l), n - 2)) + [l]
    tm = [1] * n
    for _ in range(100 - n):                     # phân bổ để Σ time = 100
        tm[random.randrange(n)] += 1
    t = _time.time(); a = Solution().minTravelTime(l, n, k, pos, tm); t1 = _time.time() - t
    b = min_travel_memo(l, n, k, pos, tm)
    assert a == b
    print(f"l={l}: đáp án {a} | {t1 * 1000:.1f} ms")

l=65262: đáp án 105906 | 5.9 ms
l=53913: đáp án 91455 | 5.9 ms
l=44007: đáp án 68280 | 5.3 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi thời gian của biển giữ lại vẫn là `time[j]` gốc (quên cộng dồn)

In [15]:
def wrong_no_accumulate(l, n, k, position, time):
    best = float('inf')
    for removed in combinations(range(1, n - 1), k):
        kept = [j for j in range(n) if j not in set(removed)]
        best = min(best, sum((position[kept[s + 1]] - position[kept[s]]) * time[kept[s]]
                             for s in range(len(kept) - 1)))       # SAI
    return best

case = examples[0][:5]
print('sai :', wrong_no_accumulate(*case), '| đúng:', Solution().minTravelTime(*case))

sai : 46 | đúng: 62


**Vì sao sai:** gộp cộng `time[i]` vào biển kế tiếp; bỏ qua làm việc xóa biển "miễn phí".

### Lỗi 2 — Cộng dồn sang biển **bên trái** (nhầm chiều gộp)

In [16]:
def wrong_left(l, n, k, position, time):
    best = float('inf')
    for removed in combinations(range(1, n - 1), k):
        rs = set(removed)
        kept = [j for j in range(n) if j not in rs]
        total = 0
        for s in range(len(kept) - 1):
            j, nxt = kept[s], kept[s + 1]
            T = sum(time[j:nxt])                 # SAI: dồn khối bị xóa phía SAU vào j
            total += (position[nxt] - position[j]) * T
        best = min(best, total)
    return best

for case in (examples[0][:5], examples[1][:5]):
    print('sai :', wrong_left(*case), '| đúng:', Solution().minTravelTime(*case))

sai : 92 | đúng: 62
sai : 37 | đúng: 34


**Vì sao sai:** đề nói `time[i+1] = time[i] + time[i+1]` rồi xóa biển `i` — thời gian đi về **bên phải**.

### Lỗi 3 — Cho phép xóa biển `0` hoặc biển cuối

In [17]:
def wrong_remove_ends(l, n, k, position, time):
    best = float('inf')
    for removed in combinations(range(0, n - 1), k):        # SAI: cho cả 0
        rs = set(removed)
        kept = [j for j in range(n) if j not in rs]
        if kept[0] != 0:
            # "biển" đầu bị xóa -> đoạn đầu không có giá -> coi như chi phí 0 cho đoạn đó
            kept = [0] + kept
        best = min(best, cost_of_kept(position, time, kept))
    return best

case = (10, 3, 1, [0, 5, 10], [9, 1, 1])
print('sai :', wrong_remove_ends(*case), '| đúng:', Solution().minTravelTime(*case))

sai : 50 | đúng: 90


**Vì sao sai:** chỉ được gộp `i` với `i > 0` và `i + 1 < n` ⇒ chỉ biển **bên trong** `1..n−2` bị xóa.

### Lỗi 4 — Thực hiện "tối đa k" thay vì "đúng k" lần gộp

In [18]:
def wrong_at_most(l, n, k, position, time):
    return min(min_travel_dp(l, n, j, position, time) for j in range(k + 1))

random.seed(41)
case = None
for _ in range(2000):
    c = rand_case(random.randint(4, 7), 0)
    c = (c[0], c[1], random.randint(1, c[1] - 2), c[3], c[4])
    if wrong_at_most(*c) != Solution().minTravelTime(*c):
        case = c
        break
print('ca thử (l, n, k, position, time):', case)
print('sai :', wrong_at_most(*case), '| đúng:', Solution().minTravelTime(*case))

ca thử (l, n, k, position, time): (5, 5, 1, [0, 1, 2, 4, 5], [8, 7, 4, 1, 2])
sai : 24 | đúng: 27


**Vì sao sai:** gộp có thể làm **tăng** thời gian (biển sau mang tổng lớn đi một quãng dài); đề bắt buộc đúng `k` lần.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Thao tác gộp phần tử kề, giá trị dồn sang một phía" ⇒ thường kết quả chỉ phụ thuộc **tập bị xóa** ⇒ DP chọn phần tử giữ lại.
- Chi phí của phần tử giữ lại phụ thuộc **phần tử giữ lại trước nó** ⇒ thêm chỉ số "trước" vào trạng thái.
- "Đúng `k` lần" với `k` nhỏ ⇒ chiều DP thứ ba `c ≤ k`, cắt bước nhảy `t − j − 1 ≤ k − c`.
- Tổng một đoạn liên tiếp ⇒ tổng tiền tố.

## 15. Biến thể của bài

1. "Tối đa `k`" lần gộp ⇒ lấy min trên mọi `c ≤ k` ở trạng thái cuối.
2. Gộp dồn sang **trái** ⇒ biển `j` mang khối bị xóa **sau** nó; trạng thái cần biết biển giữ lại **tiếp theo** (DP từ phải sang trái).
3. Chi phí gộp có thêm phí cố định mỗi lần ⇒ cộng `k · phí` (không đổi lời giải) hoặc phí phụ thuộc biển ⇒ cộng vào chuyển trạng thái.
4. Bài liên quan: LC 1000 *Minimum Cost to Merge Stones*, LC 1335 *Minimum Difficulty of a Job Schedule* (DP chia đoạn).

In [19]:
# Biến thể 1: tối đa k lần gộp
def min_travel_at_most(l, n, k, position, time):
    pre = [0]
    for t in time:
        pre.append(pre[-1] + t)
    INF = float('inf')
    dp = [dict() for _ in range(n)]
    dp[0][(-1, 0)] = 0
    for j in range(n - 1):
        for (i, c), v in dp[j].items():
            T = pre[j + 1] - pre[i + 1]
            for t in range(j + 1, min(n, j + k - c + 2)):
                key = (j, c + t - j - 1)
                cost = v + (position[t] - position[j]) * T
                if cost < dp[t].get(key, INF):
                    dp[t][key] = cost
    return min(dp[n - 1].values())

print(min_travel_at_most(*case), '(tối đa k) vs', Solution().minTravelTime(*case), '(đúng k) trên ca thử ở Lỗi 4')

24 (tối đa k) vs 27 (đúng k) trên ca thử ở Lỗi 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Mô phỏng mọi dãy gộp | `O((n−2)^k n)` | `O(kn)` | Oracle nguyên văn đề |
| 2 | Thử mọi tập bị xóa | `O(C(n−2,k) n)` | `O(n)` | Oracle nhanh hơn |
| 3 | DP có nhớ | `O(n^3 k)` | `O(n^2 k)` | Dễ viết |
| 4 | DP bottom-up có cắt | `O(n k^3)` | `O(n k^2)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `minTravelTime(4, 3, 1, [0, 1, 4], [2, 3, 7])` = ?

**Bài 2 (tính tay):** ví dụ 2 nhưng `k = 2`.

**Bài 3 (điền TODO):** viết `merged_times(time, kept)` trả về thời gian mới của các biển giữ lại.

**Bài 4 (tự viết lại):** biến thể 2 — gộp dồn sang **trái** (`time[i−1] += time[i]`, xóa `i`, với `0 < i < n−1`), viết DP và kiểm chứng bằng mô phỏng.

**Bài 5 (mini challenge):** truy vết **tập biển bị xóa** tối ưu và một **thứ tự thao tác** hợp lệ, rồi mô phỏng lại.

In [20]:
def merged_times_todo(time, kept):
    # TODO
    raise NotImplementedError

def min_travel_left_todo(l, n, k, position, time):
    # TODO
    raise NotImplementedError

def best_removal_todo(l, n, k, position, time):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Bắt buộc xóa biển 1. (2) Còn `[0, 4]`, time `[2, 10]`. (3) `4 · 2 = 8`.

**Bài 2.** (1) Thử các cặp biển bên trong. (2) Dùng `min_travel_subsets` để kiểm. (3) So với DP.

**Bài 3.** (1) Biển đầu: `time[0]`. (2) Biển `kept[s]`: tổng `time[kept[s−1]+1 .. kept[s]]`. (3) So với mô phỏng gộp.

**Bài 4.** (1) Biển giữ lại `j` mang khối bị xóa **sau** nó (tới biển giữ lại kế tiếp, không tính biển đó). (2) Khi chuyển `j → t`, chi phí `(position[t] − position[j]) · Σ time[j..t−1]` — không cần biết biển trước! (3) DP 2 chiều `(j, c)`.

**Bài 5.** (1) Lưu cha của mỗi trạng thái. (2) Tập bị xóa = các biển không nằm trên đường truy vết. (3) Xóa từ trái sang phải; sau mỗi lần chỉ số dịch đi 1.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
a1 = Solution().minTravelTime(4, 3, 1, [0, 1, 4], [2, 3, 7])
a2 = Solution().minTravelTime(5, 5, 2, [0, 1, 2, 3, 5], [8, 3, 9, 3, 3])
assert a1 == 8 == min_travel_simulate(4, 3, 1, [0, 1, 4], [2, 3, 7])
assert a2 == min_travel_simulate(5, 5, 2, [0, 1, 2, 3, 5], [8, 3, 9, 3, 3])

def merged_times(time, kept):
    out = [time[0]]
    for s in range(1, len(kept)):
        out.append(sum(time[kept[s - 1] + 1: kept[s] + 1]))
    return out

pos, tm = [0, 1, 2, 3, 5], [8, 3, 9, 3, 3]
p2, t2 = merge_at(pos, tm, 1)        # xóa biển 1
p3, t3 = merge_at(p2, t2, 2)         # xóa biển 3 gốc (chỉ số 2 sau lần xóa đầu)
assert t3 == merged_times(tm, [0, 2, 4])
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3:', merged_times(tm, [0, 2, 4]))

Bài 1: 8 | Bài 2: 20 | Bài 3: [8, 12, 6]


In [22]:
# Bài 4: gộp dồn sang trái
def merge_left_at(pos, tm, i):
    tm2 = tm[:i - 1] + [tm[i - 1] + tm[i]] + tm[i + 1:]
    return pos[:i] + pos[i + 1:], tm2

def simulate_left(l, n, k, position, time):
    best = [float('inf')]
    def rec(pos, tm, left):
        if left == 0:
            best[0] = min(best[0], travel(pos, tm)); return
        for i in range(1, len(pos) - 1):
            rec(*merge_left_at(pos, tm, i), left - 1)
    rec(position, time, k)
    return best[0]

def min_travel_left(l, n, k, position, time):
    pre = [0]
    for t in time:
        pre.append(pre[-1] + t)
    INF = float('inf')
    dp = [[INF] * (k + 1) for _ in range(n)]
    dp[0][0] = 0
    for j in range(n - 1):
        for c in range(k + 1):
            if dp[j][c] == INF:
                continue
            for t in range(j + 1, min(n, j + k - c + 2)):
                nc = c + t - j - 1
                T = pre[t] - pre[j]                 # time[j..t-1] dồn vào j
                dp[t][nc] = min(dp[t][nc], dp[j][c] + (position[t] - position[j]) * T)
    return dp[n - 1][k]

random.seed(4)
for _ in range(200):
    n = random.randint(2, 7); k = random.randint(0, min(n - 2, 3))
    case = rand_case(n, k)
    assert min_travel_left(*case) == simulate_left(*case)
print('Bài 4: gộp sang trái khớp mô phỏng')

Bài 4: gộp sang trái khớp mô phỏng


In [23]:
# Bài 5: truy vết tập bị xóa và thứ tự thao tác
def best_removal(l, n, k, position, time):
    pre = [0]
    for t in time:
        pre.append(pre[-1] + t)
    INF = float('inf')
    dp = [dict() for _ in range(n)]
    parent = {}
    dp[0][(-1, 0)] = 0
    for j in range(n - 1):
        for (i, c), v in dp[j].items():
            T = pre[j + 1] - pre[i + 1]
            for t in range(j + 1, min(n, j + k - c + 2)):
                key = (j, c + t - j - 1)
                cost = v + (position[t] - position[j]) * T
                if cost < dp[t].get(key, INF):
                    dp[t][key] = cost
                    parent[(t, key)] = (j, (i, c))
    key = min((st for st in dp[n - 1] if st[1] == k), key=lambda st: dp[n - 1][st])
    best = dp[n - 1][key]
    kept, node = [n - 1], (n - 1, key)
    while node in parent:
        j, st = parent[node]
        kept.append(j)
        node = (j, st)
    kept.reverse()
    removed = [x for x in range(n) if x not in set(kept)]
    # thứ tự thao tác: xóa từ trái sang phải; chỉ số hiện tại = chỉ số gốc - số đã xóa trước đó
    ops = [r - idx for idx, r in enumerate(removed)]
    pos, tm = position, time
    for i in ops:
        pos, tm = merge_at(pos, tm, i)
    assert travel(pos, tm) == best
    return best, removed, ops

print(best_removal(*examples[1][:5]))
print(best_removal(5, 5, 2, [0, 1, 2, 3, 5], [8, 3, 9, 3, 3]))

(34, [1], [1])
(20, [2, 3], [2, 2])


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
pre = tổng tiền tố của time
dp[0][(-1, 0)] = 0                         # (biển giữ lại trước, số đã xóa)
for j: for (i, c), v in dp[j]:
    T = pre[j+1] - pre[i+1]                # thời gian đã gộp của biển j
    for t in j+1 .. j+1+(k-c):             # giữ t, xóa t-j-1 biển
        dp[t][(j, c+t-j-1)] = min(..., v + (pos[t]-pos[j]) * T)
đáp án = min dp[n-1][(i, k)]
```

**Checklist:**

- [ ] Thời gian dồn sang **phải** (biển giữ lại mang khối bị xóa phía trước nó)?
- [ ] Chỉ xóa biển `1..n−2`?
- [ ] **Đúng** `k` lần (không phải tối đa)?
- [ ] Biển cuối không cần thời gian?
- [ ] Đối chiếu với mô phỏng nguyên văn trên dữ liệu nhỏ?